# 03 – Traffic Accident Severity Model Training

## 1. Introduction

This notebook develops a machine-learning model for the Smart Traffic Risk Prediction System using accident data collected from selected Indian highways.

The objectives are to:

* Load and validate the prepared dataset.
* Identify the target variable and input features.
* Split the data into training and testing sets.
* Train an XGBoost classification model.
* Evaluate the model using appropriate classification metrics.
* Save the trained model for integration into the application.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the prepared dataset
file_path = Path("C:/Minor Projects/dataa/processed/accident_data_prepared.csv")

if not file_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {file_path.resolve()}"
    )

df_model = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df_model.shape)

display(df_model.head())

Dataset loaded successfully!
Dataset shape: (8116, 20)


,Date,Day_of_Week,Time_of_Accident,Accident_Location_A,Accident_Location_A_Chainage_km,Accident_Location_A_Chainage_km_RoadSide,Accident_Severity_C,Causes_D,Road_Feature_E,Road_Condition_F,Weather_Conditions_H,Vehicle_Type_Involved_J_V1,Vehicle_Type_Involved_J_V2,Date_parsed,Time_parsed,Accident_Year,Accident_Month,Accident_DayOfWeek,Accident_Hour,Is_Weekend
0,1/9/2013,7,13:00,2,187.6,1,3,2,2,3,8,1,NaN,2013-09-01,1900-01-01 13:00:00,2013,9,6,13,1
1,4/9/2013,3,15:25,2,152.3,2,4,3,2,6,12,5,NaN,2013-09-04,1900-01-01 15:25:00,2013,9,2,15,0
2,9/9/2013,1,0:40,2,177.8,1,4,3,2,6,12,5,NaN,2013-09-09,1900-01-01 00:40:00,2013,9,0,0,0
3,11/9/2013,3,7:30,2,186.8,2,3,2,2,3,8,1,NaN,2013-09-11,1900-01-01 07:30:00,2013,9,2,7,0
4,17/09/2013,2,13:00,2,173.4,2,4,4,3,4,7,1,NaN,2013-09-17,1900-01-01 13:00:00,2013,9,1,13,0


## 2. Dataset Validation

Before model training, we inspect the dataset columns, data types, missing values, and target distribution. This ensures that the data is suitable for classification and helps identify issues that must be addressed before training.

In [3]:
print("Column names:")
print(df_model.columns.tolist())

print("\nData types:")
print(df_model.dtypes)

print("\nMissing values in each column:")
print(
    df_model.isnull().sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nAccident severity distribution:")
print(
    df_model["Accident_Severity_C"]
    .value_counts(dropna=False)
    .sort_index()
)

Column names:
['Date', 'Day_of_Week', 'Time_of_Accident', 'Accident_Location_A', 'Accident_Location_A_Chainage_km', 'Accident_Location_A_Chainage_km_RoadSide', 'Accident_Severity_C', 'Causes_D', 'Road_Feature_E', 'Road_Condition_F', 'Weather_Conditions_H', 'Vehicle_Type_Involved_J_V1', 'Vehicle_Type_Involved_J_V2', 'Date_parsed', 'Time_parsed', 'Accident_Year', 'Accident_Month', 'Accident_DayOfWeek', 'Accident_Hour', 'Is_Weekend']

Data types:
Date                                            str
Day_of_Week                                   int64
Time_of_Accident                                str
Accident_Location_A                           int64
Accident_Location_A_Chainage_km             float64
Accident_Location_A_Chainage_km_RoadSide      int64
Accident_Severity_C                           int64
Causes_D                                      int64
Road_Feature_E                                int64
Road_Condition_F                              int64
Weather_Conditions_H            

## 3. Feature Selection and Target Definition

The target variable for this experiment is `Accident_Severity_C`. The model will learn to classify the recorded accident severity into one of the four original classes.

We will exclude the target and raw date/time text columns from the input features. Categorical variables will be encoded appropriately, and missing vehicle information will be handled within the preprocessing pipeline.

**Note:** This experiment predicts recorded accident severity from accident-related information. It does not, by itself, establish the probability of a future accident occurring.

In [4]:
from sklearn.model_selection import train_test_split

# Target: recorded accident severity
target_column = "Accident_Severity_C"

# Select features available in the prepared dataset
categorical_features = [
    "Day_of_Week",
    "Accident_Location_A",
    "Accident_Location_A_Chainage_km_RoadSide",
    "Causes_D",
    "Road_Feature_E",
    "Road_Condition_F",
    "Weather_Conditions_H",
    "Vehicle_Type_Involved_J_V1",
    "Vehicle_Type_Involved_J_V2",
    "Accident_Month",
    "Accident_DayOfWeek",
    "Accident_Hour",
    "Is_Weekend"
]

numerical_features = [
    "Accident_Location_A_Chainage_km",
    "Accident_Year"
]

feature_columns = categorical_features + numerical_features

# Keep only the selected inputs and target
model_data = df_model[feature_columns + [target_column]].copy()

# Separate features and target
X = model_data[feature_columns]
y = model_data[target_column]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget classes:", sorted(y.unique()))
print("\nMissing values in selected features:")
print(X.isnull().sum().sort_values(ascending=False).head(10))

# Split while preserving the proportion of each severity class
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))

Feature matrix shape: (8116, 15)
Target shape: (8116,)

Target classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Missing values in selected features:
Vehicle_Type_Involved_J_V2                  6154
Accident_Location_A                            0
Day_of_Week                                    0
Causes_D                                       0
Road_Feature_E                                 0
Road_Condition_F                               0
Accident_Location_A_Chainage_km_RoadSide       0
Weather_Conditions_H                           0
Vehicle_Type_Involved_J_V1                     0
Accident_Month                                 0
dtype: int64

Training records: 6492
Testing records: 1624


## 4. Preprocessing Pipeline and XGBoost Model

A preprocessing pipeline will handle missing values and encode categorical features. We will then train an XGBoost classifier to predict the four recorded severity classes.

Using a pipeline ensures that preprocessing is fitted on the training data and then applied to the test data without learning from the test set.

In [5]:
from sklearn.preprocessing import LabelEncoder

# Convert severity labels 1, 2, 3, 4 to 0, 1, 2, 3
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Original severity labels:", label_encoder.classes_)
print("Encoded training labels:", sorted(set(y_train_encoded)))
print("Encoded testing labels:", sorted(set(y_test_encoded)))

Original severity labels: [1 2 3 4]
Encoded training labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Encoded testing labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from xgboost import XGBClassifier

# Convert categorical features to strings while preserving missing values
def convert_to_string(X):
    return X.astype(object).where(pd.notna(X), np.nan).astype(str)

categorical_pipeline = Pipeline([
    (
        "string_conversion",
        FunctionTransformer(convert_to_string, feature_names_out="one-to-one")
    ),
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Missing"
        )
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("numerical", numerical_pipeline, numerical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            random_state=42,
            n_jobs=-1
        )
    )
])

# Train the model
model.fit(X_train, y_train_encoded)

print("XGBoost model training completed successfully!")

XGBoost model training completed successfully!


## 5. Model Evaluation

We will evaluate the trained model on the held-out test dataset. Accuracy, precision, recall, F1-score, and the confusion matrix will help us understand how well the model predicts each severity class.

In [7]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Predict encoded labels
y_pred_encoded = model.predict(X_test)

# Convert predictions back to original severity labels
y_pred = label_encoder.inverse_transform(
    y_pred_encoded.astype(int)
)

print(
    "Test Accuracy:",
    round(accuracy_score(y_test, y_pred), 4)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[1, 2, 3, 4],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred,
        labels=[1, 2, 3, 4]
    )
)

Test Accuracy: 0.6533

Classification Report:
              precision    recall  f1-score   support

           1       0.95      0.27      0.43       142
           2       0.63      0.69      0.66       581
           3       0.64      0.73      0.68       618
           4       0.71      0.60      0.65       283

    accuracy                           0.65      1624
   macro avg       0.73      0.57      0.60      1624
weighted avg       0.68      0.65      0.65      1624


Confusion Matrix:
[[ 39  52  46   5]
 [  1 401 146  33]
 [  1 136 450  31]
 [  0  47  65 171]]


## 11. Save the Trained Model

Save the trained XGBoost pipeline and label encoder so they can be reused by the traffic risk prediction application without retraining the model.

In [8]:
import joblib
from pathlib import Path

# Create the models directory if it does not exist
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

# Save the complete preprocessing and prediction pipeline
joblib.dump(model, model_dir / "traffic_severity_model.pkl")

# Save the label encoder used for training
joblib.dump(label_encoder, model_dir / "severity_label_encoder.pkl")

print("Model saved successfully!")
print("Model path:", (model_dir / "traffic_severity_model.pkl").resolve())
print("Label encoder saved successfully!")
print("Encoder path:", (model_dir / "severity_label_encoder.pkl").resolve())

Model saved successfully!
Model path: C:\Minor Projects\models\traffic_severity_model.pkl
Label encoder saved successfully!
Encoder path: C:\Minor Projects\models\severity_label_encoder.pkl


In [9]:
import joblib
from pathlib import Path

model_path = Path("../models/traffic_severity_model.pkl")
encoder_path = Path("../models/severity_label_encoder.pkl")

# Load the saved files
loaded_model = joblib.load(model_path)
loaded_encoder = joblib.load(encoder_path)

print("Model loaded successfully!")
print("Label encoder loaded successfully!")
print("Severity classes:", loaded_encoder.classes_)

# Test prediction using one record from the test dataset
sample = X_test.iloc[[0]]
prediction_encoded = loaded_model.predict(sample).astype(int)
prediction = loaded_encoder.inverse_transform(prediction_encoded)

print("Actual severity:", y_test.iloc[0])
print("Predicted severity:", prediction[0])

Model loaded successfully!
Label encoder loaded successfully!
Severity classes: [1 2 3 4]
Actual severity: 4
Predicted severity: 4
